# The Scientist — Colab smoke test and Environment Hub release

This notebook installs the environment, verifies it, renders a hidden Level-3 universe using the available GPU, builds the wheel, and pushes a **private** release to the Prime Intellect Environment Hub.

Before the final cell, add `PRIME_API_KEY` to Colab Secrets or the runtime environment. The key is never printed.

In [ ]:
# @title 1. Locate the environment source
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

ENV_DIR = Path('/content/the-scientist')
SOURCE_ARCHIVE = Path('/content/the-scientist-source.zip')
REPO_URL = 'https://github.com/ritwikraha/markov-chainsaw.git'

if SOURCE_ARCHIVE.exists():
    if ENV_DIR.exists():
        shutil.rmtree(ENV_DIR)
    ENV_DIR.mkdir(parents=True)
    with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
        archive.extractall(ENV_DIR)
elif not (ENV_DIR / 'pyproject.toml').exists():
    repo_dir = Path('/content/markov-chainsaw')
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
    ENV_DIR = repo_dir / 'prime-rl' / 'the-scientist'

assert (ENV_DIR / 'pyproject.toml').exists(), f'Environment source not found at {ENV_DIR}'
print(f'Environment source: {ENV_DIR}')

In [ ]:
# @title 2. Install Prime CLI, the environment, and test dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'prime==0.6.24'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ENV_DIR), 'pytest', 'matplotlib'], check=True)
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))
print('Installed The Scientist and Prime CLI.')

In [ ]:
# @title 3. Confirm and exercise the accelerator
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={device}')
if torch.cuda.is_available():
    print(f'gpu={torch.cuda.get_device_name(0)}')
    probe = torch.randn(2048, 2048, device=device)
    checksum = (probe @ probe.T).mean().item()
    print(f'gpu_smoke_checksum={checksum:.6f}')
else:
    print('No GPU was allocated; environment verification can still run on CPU.')

In [ ]:
# @title 4. Run the environment verification suite
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(ENV_DIR / 'tests')],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

In [ ]:
# @title 5. Render a Level-3 universe and six laboratory observations
import json
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from the_scientist.environment import decode_polynomial, generate_universe

universe = generate_universe(level=3, seed=42, budget=6, domain=(-3, 3))
poly = decode_polynomial(universe['coefficients'])
axis = torch.linspace(-3, 3, 81, device=device)
x1, x2 = torch.meshgrid(axis, axis, indexing='ij')
y = torch.zeros_like(x1)
for (p1, p2), coefficient in poly.items():
    y = y + coefficient * x1.pow(p1) * x2.pow(p2)

queries = torch.tensor([[-3, -3], [-3, 3], [3, -3], [3, 3], [0, 0], [1, -1]], dtype=torch.float32, device=device)
observed_y = torch.zeros(queries.shape[0], device=device)
for (p1, p2), coefficient in poly.items():
    observed_y = observed_y + coefficient * queries[:, 0].pow(p1) * queries[:, 1].pow(p2)

x1_cpu, x2_cpu, y_cpu = x1.cpu().numpy(), x2.cpu().numpy(), y.cpu().numpy()
q_cpu, observed_cpu = queries.cpu().numpy(), observed_y.cpu().numpy()
fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(x1_cpu, x2_cpu, y_cpu, cmap='viridis', alpha=0.72, linewidth=0)
ax.scatter(q_cpu[:, 0], q_cpu[:, 1], observed_cpu, color='crimson', s=70, label='six experiments')
ax.set(xlabel='x1', ylabel='x2', zlabel='y', title='The Scientist: hidden Level-3 universe')
ax.legend()
fig.colorbar(surface, ax=ax, shrink=0.55, pad=0.1)
RENDER_PATH = Path('/content/the_scientist_level3.png')
plt.savefig(RENDER_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Hidden law (developer view only): y = {universe["equation"]}')
print(f'Render saved to {RENDER_PATH}')

In [ ]:
# @title 6. Build the Environment Hub wheel
subprocess.run([sys.executable, '-m', 'build', '--wheel', '--outdir', str(ENV_DIR / 'dist'), str(ENV_DIR)], check=True)
wheels = sorted((ENV_DIR / 'dist').glob('*.whl'))
assert wheels, 'Wheel build did not produce an artifact'
print(f'Built: {wheels[-1]}')

In [ ]:
# @title 7. Push a private release to Prime Intellect Environment Hub
def load_prime_api_key():
    key = os.environ.get('PRIME_API_KEY')
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get('PRIME_API_KEY')
    except Exception:
        return None

prime_api_key = load_prime_api_key()
if not prime_api_key:
    print('PUSH_PENDING: add PRIME_API_KEY to Colab Secrets, then rerun this cell.')
else:
    subprocess.run(['prime', 'config', 'set-api-key', prime_api_key, '--plain'], check=True, capture_output=True, text=True)
    push = subprocess.run(
        ['prime', 'env', 'push', '--path', str(ENV_DIR), '--visibility', 'PRIVATE', '--plain'],
        check=False,
        text=True,
        capture_output=True,
    )
    print(push.stdout)
    if push.stderr:
        print(push.stderr)
    if push.returncode != 0:
        raise RuntimeError(f'Prime Hub push failed with exit code {push.returncode}; see output above.')